# 01 — Chatbot con Claude API

> **Bloque:** Casos de uso | **Nivel:** Práctico
>
> Notebook complementario al tutorial [01-chatbot-claude-api.md](../../casos-de-uso/01-chatbot-claude-api.md)

Construiremos un chatbot interactivo directamente en este notebook.

In [ ]:
# %pip install anthropic python-dotenv ipywidgets

import anthropic
import os
import json
from datetime import datetime
from pathlib import Path
from IPython.display import display, HTML

client = anthropic.Anthropic()
print('✓ Listo')

## 1. Clase Chatbot

In [ ]:
class Chatbot:
    """Chatbot conversacional con Claude."""

    def __init__(self, modelo='claude-haiku-4-5-20251001',
                 system_prompt='Eres un asistente útil. Responde siempre en español.',
                 max_tokens=512, temperatura=0.7,
                 max_historial=20):
        self.modelo = modelo
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self.temperatura = temperatura
        self.max_historial = max_historial
        self.historial = []
        self.tokens_totales = {'entrada': 0, 'salida': 0}

    def chat(self, mensaje: str) -> str:
        self.historial.append({'role': 'user', 'content': mensaje})
        if len(self.historial) > self.max_historial:
            self.historial = self.historial[-self.max_historial:]

        respuesta = client.messages.create(
            model=self.modelo,
            max_tokens=self.max_tokens,
            temperature=self.temperatura,
            system=self.system_prompt,
            messages=self.historial
        )
        texto = respuesta.content[0].text
        self.historial.append({'role': 'assistant', 'content': texto})
        self.tokens_totales['entrada'] += respuesta.usage.input_tokens
        self.tokens_totales['salida'] += respuesta.usage.output_tokens
        return texto

    def stats(self):
        total = self.tokens_totales
        coste = (total['entrada'] * 0.00000025 + total['salida'] * 0.00000125)
        print(f'Mensajes: {len(self.historial)} | '
              f'Tokens: {total["entrada"]}↑ {total["salida"]}↓ | '
              f'Coste aprox: ${coste:.6f}')

    def limpiar(self):
        self.historial = []
        print('Historial limpio.')

    def guardar(self, ruta='conversacion.json'):
        datos = {
            'fecha': datetime.now().isoformat(),
            'modelo': self.modelo,
            'mensajes': self.historial
        }
        Path(ruta).write_text(json.dumps(datos, ensure_ascii=False, indent=2))
        print(f'Guardado en {ruta}')

print('✓ Clase Chatbot definida')

## 2. Chatbot genérico

In [ ]:
bot = Chatbot()

respuesta = bot.chat('Hola, ¿qué puedes hacer por mí?')
print('Bot:', respuesta)

respuesta = bot.chat('¿Cuáles son las principales librerías de Python para IA?')
print('\nBot:', respuesta)

bot.stats()

## 3. Chatbot especializado

In [ ]:
bot_ia = Chatbot(
    system_prompt="""Eres un tutor experto en Inteligencia Artificial.
- Explica conceptos con analogías simples
- Cuando uses términos técnicos, explícalos brevemente
- Sugiere recursos para profundizar cuando sea relevante
- Responde siempre en español""",
    temperatura=0.5
)

preguntas = [
    '¿Qué es el overfitting y cómo se evita?',
    '¿Puedes darme un ejemplo cotidiano de ese problema?',
]

for pregunta in preguntas:
    print(f'\n--- Pregunta: {pregunta} ---')
    print(bot_ia.chat(pregunta))

bot_ia.stats()

## 4. Comparar modelos

In [ ]:
import time

pregunta_test = 'Explica la diferencia entre supervised y unsupervised learning en 2 frases.'

modelos = [
    ('claude-haiku-4-5-20251001', 'Claude Haiku 4.5'),
    ('claude-sonnet-4-6', 'Claude Sonnet 4.6'),
]

for model_id, nombre in modelos:
    inicio = time.time()
    r = client.messages.create(
        model=model_id,
        max_tokens=200,
        messages=[{'role': 'user', 'content': pregunta_test}]
    )
    elapsed = time.time() - inicio
    
    print(f'\n=== {nombre} ({elapsed:.2f}s) ===')
    print(r.content[0].text)
    print(f'Tokens: {r.usage.input_tokens}↑ {r.usage.output_tokens}↓')

## 5. Tu chatbot personalizado

Crea tu propio chatbot especializado modificando el system prompt.

In [ ]:
# Personaliza este chatbot
mi_bot = Chatbot(
    system_prompt="""Eres un asistente especializado en [TU DOMINIO].
    [Añade tus instrucciones aquí]""",
    temperatura=0.7
)

# Inicia la conversación
print(mi_bot.chat('Hola, ¿en qué puedes ayudarme?'))

In [ ]:
# Continúa la conversación aquí
print(mi_bot.chat('Tu siguiente mensaje'))

In [ ]:
# Guardar la conversación cuando termines
mi_bot.guardar('mi_conversacion.json')
mi_bot.stats()

---
**Tutorial relacionado:** [01-chatbot-claude-api.md](../../casos-de-uso/01-chatbot-claude-api.md)